#  RAG 체인 구성 - Naïve RAG 구현

### **학습 목표:**
1. Retriever의 개념과 역할을 이해한다
2. 벡터 저장소에서 다양한 검색 방법(Top-K, Threshold, MMR)을 활용할 수 있다
3. LangChain을 사용하여 RAG 파이프라인을 구성할 수 있다
4. Gradio를 활용한 스트리밍 RAG 챗봇을 구현할 수 있다

### **실습 자료**: 
- data/transformer.pdf

---

# 환경 설정 및 준비

- 필수 라이브러리: langchain-chroma, langchain-community, faiss-cpu, langchain-openai, gradio
- 환경변수: OPENAI_API_KEY 설정 필요


`(1) Env 환경변수`

In [1]:
import os
import warnings

# Tokenizers 병렬 처리 경고 억제
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# 경고 억제 (선택사항)
warnings.filterwarnings('ignore', category=UserWarning)

from dotenv import load_dotenv
load_dotenv()

True

`(2) 기본 라이브러리`

In [2]:
import os
from glob import glob

`(3) 문서 로드`

In [3]:
from langchain_community.document_loaders import PyPDFLoader

# PDF 로더 초기화
pdf_loader = PyPDFLoader('./data/transformer.pdf')

# 동기 로딩
pdf_docs = pdf_loader.load()
print(f'PDF 문서 개수: {len(pdf_docs)}')

C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_19244\3020463323.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


PDF 문서 개수: 15


`(4) 텍스트 분할`

In [4]:
from langchain_huggingface import HuggingFaceEmbeddings

# Hugging Face의 임베딩 모델 생성
embeddings_huggingface = HuggingFaceEmbeddings(model_name="BAAI/bge-m3")

# 토크나이저 직접 접근
tokenizer = embeddings_huggingface._client.tokenizer

# 토크나이저를 사용한 예시
text = "테스트 텍스트입니다."
tokens = tokenizer(text)
print(tokens)

# 토크나이저 설정 확인
print(tokenizer.model_max_length)  # 최대 토큰 길이
print(tokenizer.vocab_size)        # 어휘 크기

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

{'input_ids': [0, 153924, 239355, 5826, 5, 2], 'attention_mask': [1, 1, 1, 1, 1, 1]}
8192
250002


In [5]:
# 토큰 수를 계산하는 함수
def count_tokens(text):
    return len(tokenizer(text)['input_ids'])

# 토큰 수 계산
text = "테스트 텍스트입니다."
print(count_tokens(text))

6


In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 텍스트 분할기 생성
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,                      
    chunk_overlap=100,           
    length_function=count_tokens,         # 토큰 수를 기준으로 분할 (len은 글자수로 카운트)
    separators=["\n\n", "\n",],   # 구분자 - 재귀적으로 순차적으로 적용 
)

# 텍스트 분할
chunks = text_splitter.split_documents(pdf_docs)
print(f"생성된 텍스트 청크 수: {len(chunks)}")
print(f"각 청크의 길이: {list(len(chunk.page_content) for chunk in chunks)}")
print(f"각 청크의 토큰 수: {list(count_tokens(chunk.page_content) for chunk in chunks)}")

생성된 텍스트 청크 수: 38
각 청크의 길이: [1378, 1796, 1831, 1857, 1292, 1609, 503, 1555, 1278, 1365, 1608, 833, 1416, 1679, 999, 1764, 1604, 539, 1219, 1645, 926, 1213, 1688, 716, 1409, 1626, 624, 1411, 1438, 914, 1496, 1340, 847, 812, 470, 438, 470, 441]
각 청크의 토큰 수: [336, 415, 405, 419, 327, 424, 127, 389, 294, 382, 412, 205, 419, 417, 226, 419, 395, 149, 390, 400, 221, 356, 411, 181, 394, 405, 188, 424, 400, 278, 423, 413, 252, 178, 128, 115, 128, 111]


In [7]:
# 청크의 텍스트 확인
print(chunks[2].page_content)

1 Introduction
Recurrent neural networks, long short-term memory [13] and gated recurrent [7] neural networks
in particular, have been firmly established as state of the art approaches in sequence modeling and
transduction problems such as language modeling and machine translation [ 35, 2, 5]. Numerous
efforts have since continued to push the boundaries of recurrent language models and encoder-decoder
architectures [38, 24, 15].
Recurrent models typically factor computation along the symbol positions of the input and output
sequences. Aligning the positions to steps in computation time, they generate a sequence of hidden
states ht, as a function of the previous hidden state ht−1 and the input for position t. This inherently
sequential nature precludes parallelization within training examples, which becomes critical at longer
sequence lengths, as memory constraints limit batching across examples. Recent work has achieved
significant improvements in computational efficiency through facto

# 벡터 저장소 기반 RAG 검색기 (Retriever)



`(1) 벡터 저장소 초기화`
- chroma 사용
- cosine distance 기준으로 인덱싱 

In [8]:
from langchain_chroma import Chroma

# Chroma 벡터 저장소 생성하기
chroma_db = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings_huggingface,    # huggingface 임베딩 사용
    collection_name="db_transformer_cosine",    # 컬렉션 이름
    persist_directory="./chroma_db",
    collection_metadata = {'hnsw:space': 'cosine'}, # l2, ip, cosine 중에서 선택 
)

# 현재 저장된 컬렉션 데이터 확인
chroma_db.get()

{'ids': ['32430e8a-9fe7-4891-9aab-682f0a4645c4',
  '781a95b6-4bc3-4b87-8f7c-a0a7fa817308',
  '11523bec-771d-41ce-8ee3-73d25a62d932',
  'a985900d-de95-4428-9751-c98424fbd8b2',
  '9564cecd-fa90-4722-b28f-9db31b7ea9b3',
  '79b2e9c1-5165-4d6e-b41c-25a41af88d79',
  '7568e35b-40c1-4c45-b781-b3b1234deee9',
  '25954347-f6f4-45b0-870f-530571efab3b',
  'c475dc12-0d27-4c93-85f9-3471c2646a26',
  '7f4c1356-73d2-437f-8dbb-f1e9ff465396',
  'ec12f643-89ab-4077-8509-4ab4e53880e3',
  'c2feb5ac-8937-4f83-9fdb-7047510e7221',
  '7bd495f5-201a-41ce-a172-fcc04135f26d',
  '26b17b59-09bc-4d7c-a5b0-4e070c49cfa8',
  '586761a3-3509-4424-a82e-e6814b17b97d',
  '2a0b5b04-8bdb-4e01-aa85-573889236f4a',
  'b0ab96fd-5d85-445f-9ea5-5bb8429a85cb',
  '675179d9-ce83-465b-af27-7ba7905a28fb',
  'acb550d4-7fed-4cf0-b886-e4db509e7cd3',
  '6cc8a7db-73a7-447f-9b28-1dd3d0094ccb',
  '8f38c998-917d-4c40-a634-fb569ffc6ec2',
  '4037ffa7-3bea-4ec5-8b23-06a0e6012283',
  '8ec4a140-80a2-4712-a79d-af327a59fa20',
  'da3330e7-6c3e-4c23-a6d7-

In [9]:
chroma_db._collection.count()

38

`(2) Top K`

In [10]:
chroma_k_retriever = chroma_db.as_retriever(
    search_kwargs={"k": 2},
)

query = "대표적인 시퀀스 모델은 어떤 것들이 있나요?"
retrieved_docs = chroma_k_retriever.invoke(query)

print(f"쿼리: {query}")
print("검색 결과:")
for i, doc in enumerate(retrieved_docs, 1):
    print(f"-{i}-\n{doc.page_content[:100]}...{doc.page_content[-100:]} [출처: {doc.metadata['source']}]")
    print("-" * 100)

쿼리: 대표적인 시퀀스 모델은 어떤 것들이 있나요?
검색 결과:
-1-
1 Introduction
Recurrent neural networks, long short-term memory [13] and gated recurrent [7] neural...he Transformer allows for significantly more parallelization and can reach a new state of the art in [출처: ./data/transformer.pdf]
----------------------------------------------------------------------------------------------------
-2-
In contrast to RNN sequence-to-sequence models [37], the Transformer outperforms the Berkeley-
Parse... 2016.
[2] Dzmitry Bahdanau, Kyunghyun Cho, and Yoshua Bengio. Neural machine translation by jointly [출처: ./data/transformer.pdf]
----------------------------------------------------------------------------------------------------


`(3) 임계값 지정`
- Similarity score threshold (기준 스코어 이상인 문서를 대상으로 추출)

In [11]:
from langchain_community.utils.math import cosine_similarity

chroma_threshold_retriever = chroma_db.as_retriever(
    search_type='similarity_score_threshold',       # cosine 유사도
    search_kwargs={'score_threshold': 0.1, 'k':5},  # 0.5 이상인 문서를 추출
)

query = "대표적인 시퀀스 모델은 어떤 것들이 있나요?"
retrieved_docs = chroma_threshold_retriever.invoke(query)

print(f"쿼리: {query}")
print("검색 결과:")
for i, doc in enumerate(retrieved_docs, 1):
    score = cosine_similarity(
        [embeddings_huggingface.embed_query(query)], 
        [embeddings_huggingface.embed_query(doc.page_content)]
        )[0][0]
    print(f"-{i}-\n{doc.page_content[:100]}...{doc.page_content[-100:]} [유사도: {score}]")
    print("-" * 100)

쿼리: 대표적인 시퀀스 모델은 어떤 것들이 있나요?
검색 결과:
-1-
1 Introduction
Recurrent neural networks, long short-term memory [13] and gated recurrent [7] neural...he Transformer allows for significantly more parallelization and can reach a new state of the art in [유사도: 0.5069071121962624]
----------------------------------------------------------------------------------------------------
-2-
In contrast to RNN sequence-to-sequence models [37], the Transformer outperforms the Berkeley-
Parse... 2016.
[2] Dzmitry Bahdanau, Kyunghyun Cho, and Yoshua Bengio. Neural machine translation by jointly [유사도: 0.5020664442959599]
----------------------------------------------------------------------------------------------------
-3-
used successfully in a variety of tasks including reading comprehension, abstractive summarization,
...ive
[10], consuming the previously generated symbols as additional input when generating the next.
2 [유사도: 0.49246664351728087]
-----------------------------------------------------------

`(4) MMR(Maximal Marginal Relevance) 검색`

In [12]:
# MMR - 다양성 고려
chroma_mmr = chroma_db.as_retriever(
    search_type='mmr',
    search_kwargs={
        'k': 3,                 # 최종적으로 반환할 문서의 수
        'fetch_k': 8,           # 유사도 기준으로 먼저 가져올 후보 문서 수 (fetch_k >= k 권장)
        'lambda_mult': 0.5,     # 유사도와 다양성의 균형 (0=최대 다양성, 1=최대 유사도, 기본값=0.5)
        # lambda_mult가 낮을수록 서로 다른 내용의 문서를, 높을수록 쿼리와 유사한 문서를 우선 선택
        },
)


query = "대표적인 시퀀스 모델은 어떤 것들이 있나요?"
retrieved_docs = chroma_mmr.invoke(query)

print(f"쿼리: {query}")
print("검색 결과:")
for i, doc in enumerate(retrieved_docs, 1):
    score = cosine_similarity(
        [embeddings_huggingface.embed_query(query)], 
        [embeddings_huggingface.embed_query(doc.page_content)]
        )[0][0]
    print(f"-{i}-\n{doc.page_content[:100]}...{doc.page_content[-100:]} [유사도: {score}]")
    print("-" * 100)

쿼리: 대표적인 시퀀스 모델은 어떤 것들이 있나요?
검색 결과:
-1-
1 Introduction
Recurrent neural networks, long short-term memory [13] and gated recurrent [7] neural...he Transformer allows for significantly more parallelization and can reach a new state of the art in [유사도: 0.5069071121962624]
----------------------------------------------------------------------------------------------------
-2-
Table 1: Maximum path lengths, per-layer complexity and minimum number of sequential operations
for ...ng
corresponds to a sinusoid. The wavelengths form a geometric progression from 2π to 10000 · 2π. We [유사도: 0.4792333405925163]
----------------------------------------------------------------------------------------------------
-3-
from our models and present and discuss examples in the appendix. Not only do individual attention
h..., according to the formula:
lrate = d−0.5
model · min(step_num−0.5, step_num · warmup_steps−1.5) (3) [유사도: 0.47091674231129765]
-----------------------------------------------------------

`(5) metadata 필터링 검색`

In [13]:
# 메타데이터 확인
chunks[0].metadata

{'producer': 'pdfTeX-1.40.25',
 'creator': 'LaTeX with hyperref',
 'creationdate': '2024-04-10T21:11:43+00:00',
 'author': '',
 'keywords': '',
 'moddate': '2024-04-10T21:11:43+00:00',
 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5',
 'subject': '',
 'title': '',
 'trapped': '/False',
 'source': './data/transformer.pdf',
 'total_pages': 15,
 'page': 0,
 'page_label': '1'}

In [14]:
# 문서 객체의 metadata를 이용한 필터링
chrom_metadata = chroma_db.as_retriever(
    search_kwargs={
        'filter': {'source': './data/transformer.pdf'},
        'k': 5, 
        }
)

query = "대표적인 시퀀스 모델은 어떤 것들이 있나요?"
retrieved_docs = chrom_metadata.invoke(query)

print(f"쿼리: {query}")
print("검색 결과:")
for i, doc in enumerate(retrieved_docs, 1):
    print(f"-{i}-\n{doc.page_content} [출처: {doc.metadata['source']}]")
    print("-" * 100)

쿼리: 대표적인 시퀀스 모델은 어떤 것들이 있나요?
검색 결과:
-1-
1 Introduction
Recurrent neural networks, long short-term memory [13] and gated recurrent [7] neural networks
in particular, have been firmly established as state of the art approaches in sequence modeling and
transduction problems such as language modeling and machine translation [ 35, 2, 5]. Numerous
efforts have since continued to push the boundaries of recurrent language models and encoder-decoder
architectures [38, 24, 15].
Recurrent models typically factor computation along the symbol positions of the input and output
sequences. Aligning the positions to steps in computation time, they generate a sequence of hidden
states ht, as a function of the previous hidden state ht−1 and the input for position t. This inherently
sequential nature precludes parallelization within training examples, which becomes critical at longer
sequence lengths, as memory constraints limit batching across examples. Recent work has achieved
significant improvements i

`(6) page_content 본문 필터링 검색`

In [15]:
# page_content를 이용한 필터링
chroma_content = chroma_db.as_retriever(
    search_kwargs={
        'k': 2,
        'where_document': {'$contains': 'recurrent'},
        }
)

query = "대표적인 시퀀스 모델은 어떤 것들이 있나요?"
retrieved_docs = chroma_content.invoke(query)

print(f"쿼리: {query}")
print("검색 결과:")
for i, doc in enumerate(retrieved_docs, 1):
    print(f"-{i}-\n{doc.page_content} [출처: {doc.metadata['source']}]")
    print("-" * 100)

쿼리: 대표적인 시퀀스 모델은 어떤 것들이 있나요?
검색 결과:
-1-
1 Introduction
Recurrent neural networks, long short-term memory [13] and gated recurrent [7] neural networks
in particular, have been firmly established as state of the art approaches in sequence modeling and
transduction problems such as language modeling and machine translation [ 35, 2, 5]. Numerous
efforts have since continued to push the boundaries of recurrent language models and encoder-decoder
architectures [38, 24, 15].
Recurrent models typically factor computation along the symbol positions of the input and output
sequences. Aligning the positions to steps in computation time, they generate a sequence of hidden
states ht, as a function of the previous hidden state ht−1 and the input for position t. This inherently
sequential nature precludes parallelization within training examples, which becomes critical at longer
sequence lengths, as memory constraints limit batching across examples. Recent work has achieved
significant improvements i

# [실습 프로젝트] Naive RAG 구현 

- 각 단계별 지시사항에 따라 코드를 완성하세요. 
- 제시된 지시사항과 LangChain 문서를 참조하여 시스템을 구성합니다. 

In [ ]:
##################### 첫 번째 시도 ##########################

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
import faiss
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS
from langchain_openai import ChatOpenAI
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate


# 0. 실제 문서 불러오기
loader = PyPDFLoader("./data/동호회 운영규정(2026년 제정).pdf")
pdf_docs = loader.load()

print(len(pdf_docs))

# 1. 청크 생성
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

# 2. 임베딩
# 모델 생성
embeddings_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3"
)

embedding = embeddings_model.embed_query("test")
#print(len(embedding))

# FAISS 생성 / 빠른 속도 (6번에서는 Chroma로 진행)
dim = 1024

faiss_index = faiss.IndexFlatL2(dim)

# 벡터저장소 생성
faiss_db = FAISS(
    embedding_function=embeddings_model,
    index=faiss_index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={}
)

#print(faiss_db.index.ntotal)

chunks = text_splitter.split_documents(pdf_docs)

#print(len(chunks))

# 3. 문서 저장
# 각 청크별 UUID 생성
import uuid

doc_ids = [
    str(uuid.uuid4())
    for _ in range(len(chunks))
]

# 벡터저장소에 문서 저장
added_doc_ids = faiss_db.add_documents(
    chunks,
    ids=doc_ids
)

print(f"{len(added_doc_ids)}개의 문서가 성공적으로 저장되었습니다.")
# for i, chunk in enumerate(chunks):
#     print(f"청크 {i+1}")
#     print(chunk.metadata)
#     print(len(chunk.page_content))
#     print("-" * 50)


# 4. 문서 검색
faiss_mmr_retriever = faiss_db.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 3,
        "fetch_k": 10,
        "lambda_mult": 0.3
    }
)

query = "동호회 활동을 안하면 폐지되나요?"
retrieved_docs = faiss_mmr_retriever.invoke(query)

for doc in retrieved_docs:
    print(doc.page_content)
    print("-" * 50)

# 5. 프롬포트 생성
template = """
당신은 사내 문서 기반 질의응답 시스템입니다.

규칙
1. [컨텍스트]에 있는 내용만 사용하세요.
2. 외부 지식은 사용하지 마세요.
3. 근거가 부족하면 추측하지 마세요.
4. 답변할 수 없다면 '문서에서 확인할 수 없습니다.'라고 답변하세요.

[컨텍스트]
{context}
[질문]
{question}

[답변 형식]
핵심 답변:
...
분석 결과:
...
추가 설명:
...
"""

prompt = ChatPromptTemplate.from_template(
    template
)

# 6. LLM 생성
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

# 7. 문서 포맷
def format_docs(docs):
    return "\n\n".join(
        doc.page_content
        for doc in docs
    )

# 8. 체인 생성
rag_chain = (
    {
        "context":
            faiss_mmr_retriever
            | format_docs,

        "question":
            RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

# 9. 실행
query = "동호회 활동을 안하면 폐지되나요?"

result = rag_chain.invoke(query)

print(query)
print(result)


4


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

6개의 문서가 성공적으로 저장되었습니다.
있다. 
 
제11조【폐지 기준】 
 
다음 사항 발생 시 동호회 승인을 취소할 수 있다. 
 
1. 3개월 이상 활동 실적 없는 경우 
2. 회원 인원 수가 3명 이하로 감소하는 경우 
3. 예산 부정사용 또는 허위보고를 하는 경우 
4. 기타 회사 규정 위반 사항이 발생하는 경우 
 
제 12 조【시행일】 
이 규정은 2026년 01월 01일부터 시행한다. 
 
 
제정 2025년 12월 11일
--------------------------------------------------
[별첨 1] 동호회 신청서 
동호회 신청서 
 
동호회명  
활동분야 (예:독서,축구,등산 등) 
동호회 목적  
대표자(회장) 이름  부서  
총무 이름  부서  
회원 명단 이름 
 
부서 
 
  
  
  
  
월 정기 활동 계획  
예상 예산 항목  
 
㈜하이비젼시스템
--------------------------------------------------
4. 특정 종교·정치 활동 목적의 모임은 불가 
 
제 5 조【승인 절차】 
 
1. 동호회 대표는 동호회 신청서 *[별첨1]를 작성하여 인사총무팀에 제출한다. 
2. 인사총무팀은 타당성 검토 후 경영진 또는 담당 임원의 승인 여부를 확정한다. 
3. 승인된 동호회는 회사의 공식 동호회로 등록된다. 
 
제 6 조【조직 구성】 
 
1. 대표(회장) : 동호회 운영 총괄 
2. 총무 : 예산관리 및 활동 내역 보고 
3. 필요시, 운영위원 구성 가능 
 
제 3 장 운영 및 지원 
 
제 7 조【운영 기준】 
 
1. 동호회는 한달 1회 이상 정기 활동을 실시하여야 한다. 
2. 활동 보고서*[별첨2]과 사진을 인사총무팀에 제출해야 한다. 
3. 동아리 활동 중, 폭행, 무단 음주, 사회적 물의 등 회사 명예를 훼손하는 활동은 금지한
다.
--------------------------------------------------
동호회 활동을 안하면 폐지되나요?
핵심 

In [32]:
import gradio as gr
from typing import Iterator

# 스트리밍 응답 생성 함수
def get_streaming_response(message: str, history) -> Iterator[str]:
    
    # RAG Chain 실행 및 스트리밍 응답 생성
    response = ""
    for chunk in rag_chain.stream(message):
        if isinstance(chunk, str):
            response += chunk
            yield response

theme = gr.themes.Glass()

with gr.Blocks(theme=theme) as demo:
    gr.ChatInterface(
        fn=get_streaming_response,
        title="동호회 운영규정 QA",
        description="운영규정 PDF 기반 RAG 챗봇",
        examples=[
            "동호회 가입 조건은?",
            "회비는 얼마인가요?",
            "회원 자격 상실 조건은?"
        ]
    )

demo.launch()

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


In [25]:
import gradio as gr

print(gr.__version__)
print(gr.__file__)

6.15.2
c:\Users\HVS\modu_llm7\faq_bot\.venv\Lib\site-packages\gradio\__init__.py


# 구현 내용

1. 문서 로딩 (회사 PDF 규정 문서 및 Excel 파일 불러오기)

2. 문서 분할 (Chunking 작업 수행)

3. 임베딩 (BAAI/bge-m3 모델 사용)

4. 벡터 저장소 저장 (FAISS DB 활용)

5. 벡터 저장소 기반 검색기(Retriever) 생성

6. LLM 연결 (GPT-4o-mini 활용)

7. RAG 체인 구성 및 질의응답 구현

8. Gradio 기반 스트리밍 챗봇 구현

---

# 추가 진행사항

* PDF 문서 기반 RAG 구현
* Excel 파일 연동 테스트
* 파일명 및 파일 유형 Metadata 관리
* 향후 특정 폴더 내 문서를 자동 수집하는 형태로 확장 검토
* Word, PPT 등 다양한 문서 형식 연동 검토


In [ ]:
######## 디벨롭 해보기 - 두 번째 시도 #######
from pathlib import Path
import pandas as pd
import uuid

from langchain_core.documents import Document
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface.embeddings import HuggingFaceEmbeddings

import faiss
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS

from langchain_openai import ChatOpenAI
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate


# 0. 실제 문서 불러오기
# data 폴더 내 PDF, Excel 파일 모두 로드

all_docs = []

for file_path in Path("./data").iterdir():

    # PDF 파일
    if file_path.suffix.lower() == ".pdf":

        loader = PyPDFLoader(str(file_path))
        docs = loader.load()

        # 파일명 메타데이터 추가
        for doc in docs:
            doc.metadata["source_file"] = file_path.name
            doc.metadata["file_type"] = "pdf"

        all_docs.extend(docs)

    # Excel 파일
    elif file_path.suffix.lower() in [".xlsx", ".xls"]:

        df = pd.read_excel(file_path)

        excel_doc = Document(
            page_content=df.to_string(index=False),
            metadata={
                "source_file": file_path.name,
                "file_type": "excel"
            }
        )

        all_docs.append(excel_doc)

print(f"로드된 문서 수 : {len(all_docs)}")


# 1. 청크 생성
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks = text_splitter.split_documents(all_docs)

print(f"생성된 청크 수 : {len(chunks)}")


# 2. 임베딩
# 모델 생성
embeddings_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3"
)

embedding = embeddings_model.embed_query("test")

print(f"임베딩 차원 : {len(embedding)}")


# FAISS 생성 / 빠른 속도
dim = len(embedding)

faiss_index = faiss.IndexFlatL2(dim)

# 벡터저장소 생성
faiss_db = FAISS(
    embedding_function=embeddings_model,
    index=faiss_index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={}
)

print(f"초기 저장 문서 수 : {faiss_db.index.ntotal}")


# 3. 문서 저장
# 각 청크별 UUID 생성

doc_ids = [
    str(uuid.uuid4())
    for _ in range(len(chunks))
]

# 벡터저장소에 문서 저장
added_doc_ids = faiss_db.add_documents(
    chunks,
    ids=doc_ids
)

print(f"{len(added_doc_ids)}개의 문서가 성공적으로 저장되었습니다.")
print(f"현재 저장 문서 수 : {faiss_db.index.ntotal}")

# 저장된 청크 확인
# for i, chunk in enumerate(chunks):
#     print(f"청크 {i+1}")
#     print(chunk.metadata)
#     print(chunk.page_content[:200])
#     print("-" * 50)


# 4. 문서 검색
# 실무에서는 similarity 검색을 더 많이 사용

faiss_retriever = faiss_db.as_retriever(
    search_kwargs={
        "k": 3
    }
)

query = "동호회 활동을 안하면 폐지되나요?"

retrieved_docs = faiss_retriever.invoke(query)

print("\n검색 결과")
print("=" * 50)

for doc in retrieved_docs:

    print(f"파일명 : {doc.metadata.get('source_file')}")
    print(f"파일유형 : {doc.metadata.get('file_type')}")
    print(doc.page_content[:300])

    print("-" * 50)


# 5. 프롬프트 생성

template = """
당신은 사내 문서 기반 질의응답 시스템입니다.

규칙
1. [컨텍스트]에 있는 내용만 사용하세요.
2. 외부 지식은 사용하지 마세요.
3. 근거가 부족하면 추측하지 마세요.
4. 답변할 수 없다면 '문서에서 확인할 수 없습니다.'라고 답변하세요.

[컨텍스트]
{context}

[질문]
{question}

[답변 형식]

핵심 답변:
...

근거:
...

추가 설명:
...
"""

prompt = ChatPromptTemplate.from_template(
    template
)


# 6. LLM 생성

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)


# 7. 문서 포맷

def format_docs(docs):

    formatted_docs = []

    for doc in docs:

        source = doc.metadata.get("source_file", "Unknown")

        formatted_docs.append(
            f"[출처: {source}]\n{doc.page_content}"
        )

    return "\n\n".join(formatted_docs)


# 8. 체인 생성

rag_chain = (
    {
        "context":
            faiss_retriever
            | format_docs,

        "question":
            RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)


# 9. 실행

query = "동호회 활동을 안하면 폐지되나요?"

result = rag_chain.invoke(query)

print("\n질문")
print(query)

print("\n답변")
print(result)

로드된 문서 수 : 39
생성된 청크 수 : 260


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

임베딩 차원 : 1024
초기 저장 문서 수 : 0
260개의 문서가 성공적으로 저장되었습니다.
현재 저장 문서 수 : 260

검색 결과
파일명 : 동호회 운영규정(2026년 제정).pdf
파일유형 : pdf
있다. 
 
제11조【폐지 기준】 
 
다음 사항 발생 시 동호회 승인을 취소할 수 있다. 
 
1. 3개월 이상 활동 실적 없는 경우 
2. 회원 인원 수가 3명 이하로 감소하는 경우 
3. 예산 부정사용 또는 허위보고를 하는 경우 
4. 기타 회사 규정 위반 사항이 발생하는 경우 
 
제 12 조【시행일】 
이 규정은 2026년 01월 01일부터 시행한다. 
 
 
제정 2025년 12월 11일
--------------------------------------------------
파일명 : 동호회 운영규정(2026년 제정).pdf
파일유형 : pdf
4. 특정 종교·정치 활동 목적의 모임은 불가 
 
제 5 조【승인 절차】 
 
1. 동호회 대표는 동호회 신청서 *[별첨1]를 작성하여 인사총무팀에 제출한다. 
2. 인사총무팀은 타당성 검토 후 경영진 또는 담당 임원의 승인 여부를 확정한다. 
3. 승인된 동호회는 회사의 공식 동호회로 등록된다. 
 
제 6 조【조직 구성】 
 
1. 대표(회장) : 동호회 운영 총괄 
2. 총무 : 예산관리 및 활동 내역 보고 
3. 필요시, 운영위원 구성 가능 
 
제 3 장 운영 및 지원 
 
제 7 조【운영 기준】 
 
1. 동호회는 한달
--------------------------------------------------
파일명 : 동호회 운영규정(2026년 제정).pdf
파일유형 : pdf
동호회 운영 규정 
<㈜하이비젼시스템> 
 
제 1 조【목적】 
 
본 규정은 임직원의 건전한 여가활동을 지원하고, 조직문화 활성화 및 사내 커뮤니케이
션 증진을 위해 동호회 조직 및 운영에 관한 사항을 정함을 목적으로 한다. 
 
제 2 조【적용범위】 
 
본 규정은 회사에 재직 중인 임직원이 구성하고 활동하

In [37]:
print("저장된 벡터 수:", faiss_db.index.ntotal)

저장된 벡터 수: 260


In [39]:
import gradio as gr
from typing import Iterator

# 스트리밍 응답 생성 함수
def get_streaming_response(message: str, history) -> Iterator[str]:
    
    # RAG Chain 실행 및 스트리밍 응답 생성
    response = ""
    for chunk in rag_chain.stream(message):
        if isinstance(chunk, str):
            response += chunk
            yield response

theme = gr.themes.Glass()

with gr.Blocks(theme=theme) as demo:
    gr.ChatInterface(
        fn=get_streaming_response,
        title="사내 운영규정 QA",
        description="운영규정 기반 RAG 챗봇",
        examples=[
            "동호회 가입 조건은?",
            "의료비 지원금 안내",
            "동호회 회원 자격 상실 조건은?"
        ]
    )

demo.launch()

* Running on local URL:  http://127.0.0.1:7865
* To create a public link, set `share=True` in `launch()`.


In [19]:
# 3단계: 문서 관리
# 새로운 문서 추가
new_doc = Document(
    page_content="쿠버네티스 클러스터 운영 가이드",
    metadata={"type": "tutorial", "author": "김미영"}
)
new_id = str(uuid.uuid4())
practice_db.add_documents(documents=[new_doc], ids=[new_id])
print(f"새 문서 추가 완료: {new_id}")

# 특정 문서 삭제 (첫 번째 문서 삭제)
delete_id = doc_ids[0]
practice_db.delete(ids=[delete_id])
print(f"문서 삭제 완료: {delete_id}")

NameError: name 'Document' is not defined

`(1) 벡터 저장소 설정`
- HuggingFace에서 지원하는 BAAI/bge-m3 임베딩 모델을 사용하여 문서를 벡터화
- FAISS DB를 벡터 스토어로 사용 (IndexFlatL2 사용: 유클리드 거리)

In [ ]:
from langchain_huggingface.embeddings import HuggingFaceEmbeddings  

# Hugging Face의 임베딩 모델 생성
# 힌트: HuggingFaceEmbeddings(model_name="BAAI/bge-m3") 사용
embeddings_model = None

# 임베딩 차원 확인
embedding = embeddings_model.embed_query("test")
print(f"임베딩 차원: {len(embedding)}")

In [ ]:
# Ollama 임베딩 모델을 사용한 FAISS 벡터 저장소 생성
import faiss 
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS

# FAISS 인덱스 초기화 (유클리드 거리 사용)
dim = 1024  # 임베딩 차원
faiss_index = faiss.IndexFlatL2(dim)  

# FAISS 벡터 저장소 생성
faiss_db = None

# 저장된 문서의 갯수 확인
print(faiss_db.index.ntotal)

In [ ]:
import uuid

# 문서 id 생성
doc_ids = [str(uuid.uuid4()) for _ in range(len(chunks))]

# 문서를 벡터 저장소에 저장
# 힌트: faiss_db.add_documents(chunks, ids=doc_ids) 사용
added_doc_ids = None

# 벡터 저장소에 저장된 문서를 확인
print(f"{len(added_doc_ids)}개의 문서가 성공적으로 벡터 저장소에 추가되었습니다.")
print(added_doc_ids)

`(2) 검색기 정의`
- mmr 검색으로 상위 3개 문서 검색하는 Retriever 사용
- 다양성을 높이는 설정을 사용 

In [ ]:
# mmr 검색기 생성
# 힌트: faiss_db.as_retriever(search_type='mmr', search_kwargs={'k': 3, 'fetch_k': 10, 'lambda_mult': 0.3})
# lambda_mult를 낮게 설정하여 다양성을 높임
faiss_mmr_retriever = None

In [ ]:
# 검색 테스트 
query = "대표적인 시퀀스 모델은 어떤 것들이 있나요?"
# 힌트: faiss_mmr_retriever.invoke(query) 사용
retrieved_docs = None

print(f"쿼리: {query}")
print("검색 결과:")
for i, doc in enumerate(retrieved_docs, 1):
    print(f"-{i}-\n{doc.page_content[:100]}...{doc.page_content[-100:]}")
    print("-" * 100)

`(3) RAG 프롬프트 구성`

- 작성 기준: 
    - LangChain의 ChatPromptTemplate 클래스 사용
    - 변수 처리는 {context}, {question} 형식 사용
    - 답변은 한글로 출력되도록 프롬프트 작성
    
- 아래 템플릿 코드를 기반으로 다음 내용을 참고하여 작성합니다. 

    1. 프롬프트 구성요소:
        - 작업 지침
        - 컨텍스트 영역
        - 질문 영역
        - 답변 형식 가이드

    2. 작업 지침:
        - 컨텍스트 기반 답변 원칙
        - 외부 지식 사용 제한
        - 불확실성 처리 방법
        - 답변 불가능한 경우의 처리 방법

    3. 답변 형식:
        - 핵심 답변 섹션
        - 근거 제시 섹션
        - 추가 설명 섹션 (필요시)

    4. 제약사항 반영:
        - 답변은 사실에 기반해야 함
        - 추측이나 가정을 최소화해야 함
        - 명확한 근거 제시가 필요함
        - 구조화된 형태로 작성되어야 함

In [ ]:
# Prompt 템플릿 (예시)
from langchain_core.prompts import ChatPromptTemplate

template = """Answer the question based only on the following context.

[Context]
{context}

[Question] 
{question}

[Answer]
"""

prompt = ChatPromptTemplate.from_template(template)

In [ ]:
# Prompt 템플릿 (여기에 작성하세요)
from langchain_core.prompts import ChatPromptTemplate

template = None

prompt = ChatPromptTemplate.from_template(template)

# 템플릿 출력
prompt.pretty_print()

`(4) RAG 체인 구성`
- LangChain의 LCEL 문법을 사용
- 검색 결과를 프롬프트의 'context'로 전달하고,
- 사용자가 입력한 질문을 그래도 프롬프트의 'question'에 전달
- LLM 설정:
    - ChatOpenAI 사용 ('gpt-4o-mini' 모델)
    - temperature: 답변의 일관성을 가져가는 설정값을 사용 
    - 기타 필요한 설정 
- 출력 파서: 문자열 부분만 출력되도록 구성

In [ ]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

# LLM 설정
# 힌트: ChatOpenAI(model='gpt-4o-mini', temperature=0) 사용
llm = None

# 문서 포맷팅
def format_docs(docs):
    return "\n\n".join([f"{doc.page_content}" for doc in docs])

# RAG 체인 생성
# 힌트: {'context': faiss_mmr_retriever | format_docs, 'question': RunnablePassthrough()} | prompt | llm | StrOutputParser()
rag_chain = None

# 체인 실행
query = "대표적인 시퀀스 모델은 어떤 것들이 있나요?"
output = None

print(f"쿼리: {query}")
print("답변:")
print(output)

`(5) Gradio 스트리밍 구현`
- ChatInterface 사용
- `chain.stream()`으로 응답을 청크 단위로 스트리밍

In [ ]:
import gradio as gr
from typing import Iterator

# 스트리밍 응답 생성 함수
def get_streaming_response(message: str, history) -> Iterator[str]:
    
    # RAG Chain 실행 및 스트리밍 응답 생성
    response = ""
    for chunk in rag_chain.stream(message):
        if isinstance(chunk, str):
            response += chunk
            yield response

# Gradio 인터페이스 설정
# 힌트: gr.ChatInterface(fn=get_streaming_response, title="RAG 기반 질의응답 시스템", description="...", examples=[...])
demo = None

# 실행
demo.launch()

In [ ]:
# demo 실행 종료
demo.close()